# Fine-tuning Gemma2 on University Faculty Handbook
### Using Torchtune + QLoRA on Google Colab (T4 GPU)

**Pipeline:** PDF -> Text Extraction -> Q&A Generation -> QLoRA Fine-tuning -> Evaluation

**Before You Start:**
1. Enable GPU: Runtime -> Change runtime type -> T4 GPU
2. Accept Gemma2 license: https://huggingface.co/google/gemma-2-2b-it
3. Get HuggingFace token: https://huggingface.co/settings/tokens
4. Upload your PDF to Google Drive at: My Drive/school-llm/data/school_rules.pdf

## Step 0: Install Packages
Run this cell FIRST, before importing torch. Then RESTART the session.

In [ ]:
import subprocess
print('Installing packages...')
subprocess.run(['pip', 'uninstall', 'torchao', 'torchtune', '-y'], capture_output=True)
subprocess.run(['pip', 'install', 'torchao==0.9.0', '--no-cache-dir', '-q'], capture_output=True)
subprocess.run(['pip', 'install', 'torchtune==0.5.0', '--no-cache-dir', '-q'], capture_output=True)
subprocess.run(['pip', 'install', 'pdfplumber', '-q'], capture_output=True)
subprocess.run(['pip', 'install', 'rouge-score', '-q'], capture_output=True)
print('Done! Now RESTART: Runtime -> Restart session')
print('After restart, skip this cell and run from Step 1.')

## Step 1: Verify Environment
After restarting, start here.

In [ ]:
import torch
import torchtune
print('PyTorch:', torch.__version__)
print('torchtune:', torchtune.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
from torchtune.models.gemma2 import qlora_gemma2_2b
print('All imports successful!')

## Step 2: Mount Google Drive

In [ ]:
import os, shutil
from google.colab import drive
if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')
else:
    print('Drive already mounted!')
folders = [
    '/content/drive/MyDrive/school-llm/models',
    '/content/drive/MyDrive/school-llm/data',
    '/content/drive/MyDrive/school-llm/output',
    '/content/drive/MyDrive/school-llm/logs',
    '/content/output',
    '/content/logs',
]
for folder in folders:
    os.makedirs(folder, exist_ok=True)
total, used, free = shutil.disk_usage('/content/drive/MyDrive')
print(f'Drive - Total: {total/1e9:.1f}GB | Used: {used/1e9:.1f}GB | Free: {free/1e9:.1f}GB')
if free/1e9 < 15:
    print('WARNING: Less than 15GB free!')
else:
    print('Sufficient Drive space available')

## Step 3: Download Gemma2-2B Model
This takes 15-20 minutes. Model is saved to Drive so you only need to do this once.

In [ ]:
# Replace with your actual HuggingFace token
# Or use Colab Secrets: left sidebar key icon -> add HF_TOKEN
HF_TOKEN = 'hf_YOUR_TOKEN_HERE'
# from google.colab import userdata
# HF_TOKEN = userdata.get('HF_TOKEN')

MODEL_PATH = '/content/drive/MyDrive/school-llm/models/gemma-2-2b'
REQUIRED_FILES = ['config.json', 'tokenizer.model', 'model-00001-of-00002.safetensors']

if os.path.exists(MODEL_PATH) and all(
    os.path.exists(os.path.join(MODEL_PATH, f)) for f in REQUIRED_FILES
):
    print('Model already downloaded!')
    for f in sorted(os.listdir(MODEL_PATH)):
        size = os.path.getsize(os.path.join(MODEL_PATH, f)) / 1e6
        print(f'  {f} ({size:.1f} MB)')
else:
    from huggingface_hub import snapshot_download
    print('Downloading Gemma2-2B... (15-20 minutes)')
    snapshot_download(
        repo_id='google/gemma-2-2b-it',
        local_dir='/content/gemma-2-2b-tmp',
        token=HF_TOKEN,
        ignore_patterns=['*.msgpack', '*.h5', 'flax_model*']
    )
    print('Copying to Drive...')
    shutil.copytree('/content/gemma-2-2b-tmp', MODEL_PATH, dirs_exist_ok=True)
    print('Model saved to Drive!')

## Step 4: Extract Text from PDF

In [ ]:
import pdfplumber

PDF_PATH = '/content/drive/MyDrive/school-llm/data/school_rules.pdf'

def extract_pdf(pdf_path):
    pages = []
    with pdfplumber.open(pdf_path) as pdf:
        for i, page in enumerate(pdf.pages):
            text = page.extract_text()
            if text and text.strip():
                pages.append({'page': i + 1, 'content': text.strip()})
    return pages

if not os.path.exists(PDF_PATH):
    print('PDF not found! Upload your PDF to:', PDF_PATH)
else:
    pages = extract_pdf(PDF_PATH)
    print(f'Extracted {len(pages)} pages')
    print('--- Sample (Page 1) ---')
    print(pages[0]['content'][:500])

## Step 5: Generate Q&A Training Data
Rule-based extraction - no API needed.

In [ ]:
import json, re, random

def generate_qa_from_text(pages):
    all_qa = []
    patterns = [
        {'trigger': ['policy', 'policies', 'procedure'],
         'questions': ['What is the policy regarding {topic}?', 'What are the procedures for {topic}?']},
        {'trigger': ['eligible', 'eligibility', 'requirement'],
         'questions': ['Who is eligible for {topic}?', 'What are the requirements for {topic}?']},
        {'trigger': ['deadline', 'schedule', 'annual'],
         'questions': ['When is the deadline for {topic}?']},
        {'trigger': ['submit', 'apply', 'request'],
         'questions': ['How do I {topic}?', 'What is the process to {topic}?']},
        {'trigger': ['benefit', 'leave', 'salary'],
         'questions': ['What benefits are available for {topic}?']},
    ]
    for page in pages:
        content = page['content']
        page_qa = []
        used_questions = set()
        for pattern in patterns:
            for trigger in pattern['trigger']:
                if trigger.lower() in content.lower():
                    idx = content.lower().find(trigger)
                    topic = content[max(0,idx-20):idx+40].strip()[:50]
                    sentences = re.split(r'(?<=[.!?])\s+', content)
                    answer_sents = [s for s in sentences if trigger.lower() in s.lower()]
                    answer = ' '.join(answer_sents[:3]).strip() if answer_sents else content[:300]
                    if len(answer) < 20:
                        continue
                    question = random.choice(pattern['questions']).format(topic=topic)
                    if question not in used_questions:
                        used_questions.add(question)
                        page_qa.append({'instruction': question, 'input': '', 'output': answer})
                    if len(page_qa) >= 5:
                        break
            if len(page_qa) >= 5:
                break
        all_qa.extend(page_qa)
    return all_qa

all_qa = generate_qa_from_text(pages)
print(f'Generated {len(all_qa)} Q&A pairs')
print('--- Sample ---')
for item in random.sample(all_qa, min(3, len(all_qa))):
    print(f'Q: {item["instruction"]}')
    print(f'A: {item["output"][:150]}')
    print('-' * 50)

DATA_PATH = '/content/drive/MyDrive/school-llm/data/school_data.jsonl'
with open(DATA_PATH, 'w', encoding='utf-8') as f:
    for qa in all_qa:
        f.write(json.dumps(qa, ensure_ascii=False) + '\n')
print(f'Saved to {DATA_PATH}')

## Step 6: Create Dataset Class and Config

In [ ]:
# Create custom dataset
dataset_code = '''
from torchtune.datasets import SFTDataset
from torchtune.data import InputOutputToMessages

def school_dataset(tokenizer, data_files="school_data.jsonl", packed=False):
    return SFTDataset(
        source="json",
        data_files=data_files,
        split="train",
        message_transform=InputOutputToMessages(
            train_on_input=False,
            column_map={"input": "instruction", "output": "output"}
        ),
        model_transform=tokenizer,
    )
'''
with open('/content/custom_dataset.py', 'w') as f:
    f.write(dataset_code)
print('custom_dataset.py created')

# Auto-detect safetensors files
MODEL_PATH = '/content/drive/MyDrive/school-llm/models/gemma-2-2b'
safetensors = sorted([f for f in os.listdir(MODEL_PATH) if f.endswith('.safetensors')])
print('Model files:', safetensors)
checkpoint_files_yaml = '\n    - '.join(safetensors)

config = f"""
model:
  _component_: torchtune.models.gemma2.qlora_gemma2_2b
  lora_attn_modules: ['q_proj', 'k_proj', 'v_proj']
  apply_lora_to_mlp: True
  lora_rank: 16
  lora_alpha: 32
  lora_dropout: 0.05
  quantize_base: True

tokenizer:
  _component_: torchtune.models.gemma.gemma_tokenizer
  path: {MODEL_PATH}/tokenizer.model
  max_seq_len: 2048

checkpointer:
  _component_: torchtune.training.FullModelHFCheckpointer
  checkpoint_dir: {MODEL_PATH}
  checkpoint_files:
    - {checkpoint_files_yaml}
  output_dir: /content/output
  model_type: GEMMA2
  recipe_checkpoint: null

resume_from_checkpoint: False
save_every_n_epochs: 1
save_adapter_weights_only: True

dataset:
  _component_: custom_dataset.school_dataset
  data_files: /content/drive/MyDrive/school-llm/data/school_data.jsonl
  packed: False

seed: 42
shuffle: True
epochs: 3
max_steps_per_epoch: null
batch_size: 2
gradient_accumulation_steps: 8

optimizer:
  _component_: torch.optim.AdamW
  lr: 2e-4
  weight_decay: 0.01

lr_scheduler:
  _component_: torchtune.training.lr_schedulers.get_cosine_schedule_with_warmup
  num_warmup_steps: 100

loss:
  _component_: torchtune.modules.loss.CEWithChunkedOutputLoss

device: cuda
dtype: bf16
enable_activation_checkpointing: True
enable_activation_offloading: False
clip_grad_norm: 1.0
compile: False

metric_logger:
  _component_: torchtune.training.metric_logging.DiskLogger
  log_dir: /content/logs

log_every_n_steps: 10
log_peak_memory_stats: True
output_dir: /content/output
"""
with open('/content/school_gemma2_config.yaml', 'w') as f:
    f.write(config)
print('Config created!')

## Step 7: Run Fine-tuning
Expected time: ~10 min/epoch x 3 epochs = ~30 minutes on T4

**Do NOT close the browser tab during training!**
**Do NOT press Ctrl+C during checkpoint saving!**

In [ ]:
print('Starting fine-tuning (~30 minutes)...')
print('Watch for Loss decreasing each step - that means it is working!')
!tune run lora_finetune_single_device --config /content/school_gemma2_config.yaml

## Step 8: Save to Google Drive

In [ ]:
print('Saving trained adapter to Google Drive...')
for epoch_dir in sorted(os.listdir('/content/output')):
    epoch_path = f'/content/output/{epoch_dir}'
    if os.path.isdir(epoch_path):
        print(f'  {epoch_dir}/:', os.listdir(epoch_path))
shutil.copytree('/content/output', '/content/drive/MyDrive/school-llm/output', dirs_exist_ok=True)
print('Saved to Google Drive: My Drive/school-llm/output/')

## Step 9: Evaluation
**Restart session first to free GPU memory**: Runtime -> Restart session
Then install evaluation packages below.

In [ ]:
# Run after restarting session - BEFORE importing torch
import subprocess
subprocess.run(['pip', 'install', 'torchao>=0.16.0', '-q', '--no-cache-dir'], capture_output=True)
subprocess.run(['pip', 'install', 'transformers==4.47.0', 'peft', 'rouge-score', '-q'], capture_output=True)
print('Evaluation packages installed!')
print('Now RESTART session again: Runtime -> Restart session')

In [ ]:
# After second restart - run evaluation
import torch, os, shutil
from google.colab import drive
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')

MODEL_PATH = '/content/drive/MyDrive/school-llm/models/gemma-2-2b'
ADAPTER_PATH = '/content/drive/MyDrive/school-llm/output/epoch_2'

hf_tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)

def generate_answer(model, question, max_new_tokens=200):
    prompt = f'### Instruction:\n{question}\n\n### Response:\n'
    inputs = hf_tokenizer(prompt, return_tensors='pt').to(model.device)
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=max_new_tokens,
                                  temperature=0.3, top_p=0.9, do_sample=True)
    full = hf_tokenizer.decode(outputs[0], skip_special_tokens=True)
    return full.split('### Response:')[-1].strip() if '### Response:' in full else full.strip()

# Customize these questions for your handbook!
test_questions = [
    'What is the policy on faculty office hours?',
    'How many sick days are faculty members entitled to?',
    'What is the procedure for requesting a leave of absence?',
    'What is the travel reimbursement policy?',
    'How are faculty performance reviews conducted?',
    'What happens if a faculty member violates academic integrity?',
    'What is the outside employment policy?',
    'How do I submit a grade change request?',
]

# Base model answers
print('Loading base model...')
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH, torch_dtype=torch.bfloat16, device_map='auto', low_cpu_mem_usage=True)
base_model.eval()
base_answers = [generate_answer(base_model, q) for q in test_questions]
print('Base model done!')
del base_model; torch.cuda.empty_cache()

# Fine-tuned model answers
print('Loading fine-tuned model...')
base_hf = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH, torch_dtype=torch.bfloat16, device_map='auto', low_cpu_mem_usage=True)
ft_model = PeftModel.from_pretrained(base_hf, ADAPTER_PATH)
ft_model.eval()
ft_answers = [generate_answer(ft_model, q) for q in test_questions]
print('Fine-tuned model done!')

In [ ]:
from rouge_score import rouge_scorer
import matplotlib.pyplot as plt
import numpy as np
import json

# Replace with actual answers from YOUR handbook
ground_truth = [
    'Faculty members must hold regular office hours as specified in their contract.',
    'Faculty are entitled to sick leave per the university sick leave policy.',
    'Faculty must submit a leave request to their department chair for approval.',
    'Travel reimbursement requires pre-approval and submission of receipts.',
    'Faculty performance reviews are conducted annually by the department chair.',
    'Faculty who violate academic integrity policies are subject to disciplinary action.',
    'Outside employment must be disclosed and approved by the university.',
    'Grade change requests must be submitted through the registrar.',
]

scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)
base_scores = [scorer.score(gt, ba)['rougeL'].fmeasure for gt, ba in zip(ground_truth, base_answers)]
ft_scores   = [scorer.score(gt, fa)['rougeL'].fmeasure for gt, fa in zip(ground_truth, ft_answers)]
base_avg = sum(base_scores) / len(base_scores)
ft_avg   = sum(ft_scores)   / len(ft_scores)
improvement = ft_avg - base_avg

print('=' * 50)
print(f'Base Model   RougeL: {base_avg:.3f}')
print(f'Fine-tuned   RougeL: {ft_avg:.3f}')
print(f'Improvement:        +{improvement:.3f} ({improvement/base_avg*100:.1f}%)')
print('=' * 50)

for i, q in enumerate(test_questions):
    improved = ft_scores[i] > base_scores[i]
    print(f'Q{i+1}: {"Improved" if improved else "Regressed"} | Base: {base_scores[i]:.3f} | FT: {ft_scores[i]:.3f}')
    print(f'  [Base]     {base_answers[i][:150]}')
    print(f'  [FT]       {ft_answers[i][:150]}')

# Chart
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
x = np.arange(len(test_questions))
w = 0.35
ax1.bar(x - w/2, base_scores, w, label='Base', color='skyblue')
ax1.bar(x + w/2, ft_scores, w, label='Fine-tuned', color='orange')
ax1.set_xlabel('Questions'); ax1.set_ylabel('RougeL')
ax1.set_title('Per-Question Performance')
ax1.set_xticks(x); ax1.set_xticklabels([f'Q{i+1}' for i in range(len(test_questions))])
ax1.legend(); ax1.set_ylim(0, 1.0)
ax2.bar(['Base', 'Fine-tuned'], [base_avg, ft_avg], color=['skyblue', 'orange'], width=0.4)
ax2.set_ylabel('Average RougeL')
ax2.set_title(f'Average Performance (+{improvement/base_avg*100:.1f}%)')
ax2.set_ylim(0, 1.0)
for i, v in enumerate([base_avg, ft_avg]):
    ax2.text(i, v + 0.02, f'{v:.3f}', ha='center', fontweight='bold')
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/school-llm/eval_chart.png', dpi=150)
plt.show()

with open('/content/drive/MyDrive/school-llm/eval_results.json', 'w') as f:
    json.dump({'questions': test_questions, 'base_answers': base_answers,
               'ft_answers': ft_answers, 'base_rouge': base_avg,
               'ft_rouge': ft_avg, 'improvement_pct': improvement/base_avg*100}, f, indent=2)
print('Results saved to Google Drive!')

## Summary
What we built:
- Extracted text from faculty handbook PDF
- Generated Q&A training data automatically
- Fine-tuned Gemma2-2B with QLoRA (4-bit quantization)
- Evaluated with RougeL scores and visualized results

To improve results:
1. Use Claude/GPT to generate higher quality Q&A pairs (500+ recommended)
2. Increase epochs to 5
3. Add RAG for accurate policy retrieval